# Analysis: Recurring CTR Anomaly in Xiaohongshu Recommendations

This notebook reproduces the exploratory analysis behind the presentation. The goal is to examine why the sixth slot in repeated seven-position recommendation blocks has unusually low click-through rate.

## Research Setup

The main idea is to compare recommendation performance by absolute position and by within-block position. Commercial content is identified with `commercial_flag != 0`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset

DATA_PATH = Path("data/qilin_note_metadata_final.parquet")
REPORT_DIR = Path("reports")
FIGURE_DIR = REPORT_DIR / "figures"
REPORT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

MAX_REQUESTS = 10000

## Load and Prepare Note Metadata

In [ ]:
notes = pd.read_parquet(DATA_PATH).copy()

impressions = pd.to_numeric(notes["imp_rec_num"], errors="coerce")
clicks = pd.to_numeric(notes["click_rec_num"], errors="coerce")
notes["prior_rec_ctr"] = clicks.div(impressions.where(impressions > 0)).clip(0, 1)
notes["is_commercial"] = (notes["commercial_flag"].fillna(0) != 0).astype(int)
notes["has_video"] = (notes["video_duration"].fillna(0) > 0).astype(int)
notes["has_images"] = (notes["image_num"].fillna(0) > 0).astype(int)

notes.shape, notes.head()

In [ ]:
metadata_summary = pd.Series(
    {
        "num_notes": len(notes),
        "num_taxonomy1": notes["taxonomy1_id"].nunique(dropna=True),
        "commercial_note_rate": notes["is_commercial"].mean(),
        "median_impressions": notes["imp_rec_num"].median(),
        "median_clicks": notes["click_rec_num"].median(),
        "mean_prior_ctr": notes["prior_rec_ctr"].mean(),
    },
    name="value",
)
metadata_summary.to_csv(REPORT_DIR / "metadata_summary.csv")
metadata_summary

## Load Recommendation Logs

Qilin stores recommendation train/test data as separate configs. The Hugging Face split is still named `train` for both configs.

In [ ]:
def flatten_recommendation_requests(dataset, max_requests=None):
    rows = []
    for request_number, request in enumerate(dataset):
        if max_requests is not None and request_number >= max_requests:
            break

        details = request.get("rec_result_details_with_idx") or []
        recent_clicked = request.get("recent_clicked_note_idxs") or []
        query = request.get("query") or ""

        for item in details:
            rows.append(
                {
                    "request_idx": request.get("request_idx"),
                    "session_idx": request.get("session_idx"),
                    "user_idx": request.get("user_idx"),
                    "query": query,
                    "query_length": len(query),
                    "recent_click_count": len(recent_clicked),
                    "note_idx": item.get("note_idx"),
                    "position": item.get("position"),
                    "clicked": int((item.get("click") or 0) > 0),
                    "liked": int((item.get("like") or 0) > 0),
                    "collected": int((item.get("collect") or 0) > 0),
                    "page_time": item.get("page_time"),
                }
            )
    return pd.DataFrame(rows)


recommendation_train = load_dataset(
    "THUIR/Qilin",
    "recommendation_train",
    split="train",
    cache_dir="hf_cache",
)

rec = flatten_recommendation_requests(recommendation_train, max_requests=MAX_REQUESTS)
rec = rec.merge(
    notes[
        [
            "note_idx",
            "commercial_flag",
            "is_commercial",
            "note_type",
            "content_length",
            "image_num",
            "video_duration",
            "taxonomy1_id",
            "prior_rec_ctr",
        ]
    ],
    on="note_idx",
    how="left",
)

rec["block"] = ((rec["position"] - 1) // 7 + 1).astype(int)
rec["within_block_position"] = ((rec["position"] - 1) % 7 + 1).astype(int)
rec.shape, rec.head()

## CTR by Recommendation Position

In [ ]:
position_summary = (
    rec.groupby("position")
    .agg(
        impressions=("clicked", "size"),
        clicks=("clicked", "sum"),
        commercial_rate=("is_commercial", "mean"),
    )
    .reset_index()
)
position_summary["ctr"] = position_summary["clicks"] / position_summary["impressions"]
position_summary.to_csv(REPORT_DIR / "position_ctr.csv", index=False)
position_summary.head(20)

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(position_summary["position"], position_summary["ctr"], marker="o", label="CTR")
ax1.set_xlabel("Recommendation position")
ax1.set_ylabel("CTR")
ax1.set_title("Click-Through Rate by Recommendation Position")
for p in [6, 13, 20, 27, 34, 41]:
    ax1.axvline(p, color="#d62728", alpha=0.18)
ax1.legend(loc="upper right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "ctr_by_position.png", dpi=160)
plt.show()

## Commercial Content by Position

This checks the initial hypothesis that the sixth slot has lower CTR because it receives more commercial notes.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    position_summary["position"],
    position_summary["commercial_rate"],
    marker="o",
    color="#f28e2b",
)
ax.set_xlabel("Recommendation position")
ax.set_ylabel("Commercial content rate")
ax.set_title("Commercial Content Rate by Recommendation Position")
for p in [6, 13, 20, 27, 34, 41]:
    ax.axvline(p, color="#d62728", alpha=0.18)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "commercial_rate_by_position.png", dpi=160)
plt.show()

## Seven-Position Block Analysis

In [ ]:
slot_summary = (
    rec.groupby("within_block_position")
    .agg(
        impressions=("clicked", "size"),
        clicks=("clicked", "sum"),
        commercial_rate=("is_commercial", "mean"),
        avg_position=("position", "mean"),
    )
    .reset_index()
)
slot_summary["ctr"] = slot_summary["clicks"] / slot_summary["impressions"]
slot_summary.to_csv(REPORT_DIR / "within_block_position_summary.csv", index=False)
slot_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(slot_summary["within_block_position"], slot_summary["ctr"], color="#4c78a8")
ax.bar(
    slot_summary.loc[slot_summary["within_block_position"] == 6, "within_block_position"],
    slot_summary.loc[slot_summary["within_block_position"] == 6, "ctr"],
    color="#d62728",
)
ax.set_xlabel("Within-block position")
ax.set_ylabel("CTR")
ax.set_title("CTR by Position Within Each Seven-Item Block")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "ctr_by_within_block_position.png", dpi=160)
plt.show()

## Non-Commercial Content Check

If commercial exposure fully explained the CTR dip, the sixth-slot penalty should largely disappear after removing commercial notes.

In [ ]:
non_commercial = rec[rec["is_commercial"] == 0].copy()
non_commercial_slot_summary = (
    non_commercial.groupby("within_block_position")
    .agg(impressions=("clicked", "size"), clicks=("clicked", "sum"))
    .reset_index()
)
non_commercial_slot_summary["ctr"] = (
    non_commercial_slot_summary["clicks"] / non_commercial_slot_summary["impressions"]
)
non_commercial_slot_summary.to_csv(
    REPORT_DIR / "non_commercial_within_block_position_summary.csv",
    index=False,
)
non_commercial_slot_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(
    slot_summary["within_block_position"],
    slot_summary["ctr"],
    marker="o",
    label="All notes",
)
ax.plot(
    non_commercial_slot_summary["within_block_position"],
    non_commercial_slot_summary["ctr"],
    marker="o",
    label="Non-commercial only",
)
ax.set_xlabel("Within-block position")
ax.set_ylabel("CTR")
ax.set_title("CTR Dip After Removing Commercial Content")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "non_commercial_ctr_check.png", dpi=160)
plt.show()

## Takeaway

The sixth slot has both a commercial-content concentration and a lower engagement pattern. The dip remains visible after removing commercial content, so commercial exposure alone does not fully explain the anomaly.